In [1]:
import streamlit as st
import pandas as pd
import requests as rq
import bs4
import plotly.express as px
import re

In [2]:
url = "https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
page = rq.get(url, headers=headers)
bs4page = bs4.BeautifulSoup(page.text, "html.parser")
tables = bs4page.find("table", {"class": "wikitable"})
gdp = pd.read_html(str(tables))[0]
gdp = gdp.dropna(how='all')
gdp.columns = ['Area', 'IMF', 'World Bank', 'UN']

C:\Users\49498\AppData\Local\Temp\ipykernel_17824\1536500500.py:6: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  gdp = pd.read_html(str(tables))[0]


In [3]:
gdp

,Area,IMF,World Bank,UN
0,World,123584494,111326370,100834796
1,United States,31821293,28750956,29298000
2,China[n 1],20650754,18743803,18743802
3,Germany,5328184,4685593,4659929
4,India,4515629,3909892,3952244
...,...,...,...,...
217,Kiribati,343,308,343
218,Marshall Islands,332,290,281
219,Nauru,183,163,187
220,Montserrat,—N/a,—N/a,81


In [4]:
df = pd.DataFrame()
df['Area'] = gdp.iloc[:, 0]
df['IMF'] = gdp.iloc[:, 2]
df['World Bank'] = gdp.iloc[:, 4]
df['UN'] = gdp.iloc[:, 6]

IndexError: single positional indexer is out-of-bounds

In [ ]:




    df = df[~df['IMF'].astype(str).str.contains('N/a|—', case=False, na=False)]
    df['Area'] = df['Area'].astype(str).apply(lambda x: re.sub(r'\[.*?\]()', '', x).strip())
    
    for col in ['IMF', 'World Bank', 'UN']:
        df[col] = df[col].astype(str).str.split(r'\(|\[|（').str[0]
        df[col] = df[col].str.replace(r'[^\d]', '', regex=True)
        df[col] = pd.to_numeric(df[col], errors='coerce')
        
    df = df.dropna(subset=['Area']).reset_index(drop=True)
    df = df[df['Area'] != 'World']

    url_region = "https://en.wikipedia.org/wiki/List_of_countries_and_territories_by_the_United_Nations_geoscheme"
    page_region = rq.get(url_region, headers=headers)
    bs4page_region = bs4.BeautifulSoup(page_region.text, "html.parser")
    tables_region = bs4page_region.find("table", {"class": "wikitable"})
    region_table = pd.read_html(str(tables_region))[0]

    regions_country = region_table.iloc[:, 0].copy()
    regions_continent = region_table.iloc[:, 1].copy() 

    def getregion(area):
        area = area.lower()
        if 'macau' in area or 'macao' in area or 'taiwan' in area: 
            return 'Asia'
        for i, name in enumerate(regions_country):
            country_or_area = str(name)
            if area.lower() in country_or_area.lower():
                return str(regions_continent.iloc[i])
        return 'Other'

    df['Region'] = df['Area'].apply(getregion)
    return df

with st.spinner("Scraping data using original logic..."):
    df = load_data()

source = st.selectbox("Select Data Source:", ["IMF", "UN", "World Bank"])

df_plot = df.dropna(subset=[source]).sort_values(by=source, ascending=True)

fig = px.bar(
    df_plot,
    x="Region",
    y=source,
    color="Area",
    labels={source: "GDP (Million USD)", "Region": "Region", "Area": "Country"},
    barmode="stack",
)

fig.update_layout(
    showlegend=False,
    font={'family': 'Helvetica'},
    xaxis={'categoryorder': 'total descending', 'tickangle': -45},
    yaxis={'type': 'log', 'title': f'GDP (Million USD) - {source} (Log Scale)'},
    title={'text': f"GDP by Country Stacked within Regions ({source})", 'x': 0.5, 'y': 0.9, 'xanchor': 'center'}
)

st.plotly_chart(fig, use_container_width=True)